In [11]:
# Système, fichiers et utilitaires
import io
import os
import re
import sqlite3
import sys
import unicodedata
import warnings

# Masquage des avertissements de déprécation
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Réseau
import requests

# Manipulation et analyse de données
import numpy as np
import pandas as pd

# Visualisation et Data Profiling
from data_profiling import ProfileReport
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

# Machine Learning & Prétraitement (Scikit-Learn & XGBoost)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import BallTree
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

# Configuration d'affichage et de style
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")
%matplotlib inline

print("Imports OK")

Imports OK


In [12]:
# Diagnostic du jeu de données d'origine

df_raw = pd.read_csv("../data/raw/dataset_irve.csv", low_memory=False)

# Info dataset

print("=== VOLUMÉTRIE INITIALE ===")
print(f"Nombre de lignes : {df_raw.shape[0]:,}")
print(f"Nombre de colonnes : {df_raw.shape[1]}")

print("\n=== APERÇU DES DONNÉES ===")
display(df_raw.head(3))

print("\n=== TYPES ET COMPLÉTUDE ===")
df_raw.info(verbose=True, show_counts=True)

# Statistiques descriptives des variables numériques
print("\n=== STATISTIQUES DESCRIPTIVES ===")
display(df_raw.describe())

# Diagnostic des valeurs manquantes
print("\n=== SYNTHÈSE DES VALEURS MANQUANTES ===")
missing_df = pd.DataFrame(
    {
        "nuls": df_raw.isna().sum(),
        "pourcentage": (df_raw.isna().mean() * 100).round(2),
    }
).sort_values(by="nuls", ascending=False)

display(missing_df[missing_df["nuls"] > 0])

# Diagnostic complet des doublons
print("\n=== DIAGNOSTIC DES DOUBLONS ===")

# Doublons stricts (lignes 100 % identiques)
exact_dup = df_raw.duplicated().sum()
print(f"1. Doublons stricts (lignes entièrement identiques) : {exact_dup}")

# Doublons sur le Point de Charge (id_pdc_itinerance)
if "id_pdc_itinerance" in df_raw.columns:
    pdc_dup = df_raw.duplicated(subset=["id_pdc_itinerance"]).sum()
    print(f"2. Doublons sur l'identifiant PDC (id_pdc_itinerance) : {pdc_dup}")

# C. Analyse par station (identifier si une station contient plusieurs PDC)
if "id_station_itinerance" in df_raw.columns:
    unique_stations = df_raw["id_station_itinerance"].nunique()
    print(
        f"3. Nombre de stations uniques (id_station_itinerance) : {unique_stations:,}"
    )

=== VOLUMÉTRIE INITIALE ===
Nombre de lignes : 226,963
Nombre de colonnes : 52

=== APERÇU DES DONNÉES ===


,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,implantation_station,adresse_station,code_insee_commune,coordonneesXY,nbre_pdc,id_pdc_itinerance,id_pdc_local,puissance_nominale,prise_type_ef,prise_type_2,prise_type_combo_ccs,prise_type_chademo,prise_type_autre,gratuit,paiement_acte,paiement_cb,paiement_autre,tarification,condition_acces,reservation,horaires,accessibilite_pmr,restriction_gabarit,station_deux_roues,raccordement,num_pdl,date_mise_en_service,observations,date_maj,cable_t2_attache,last_modified,datagouv_dataset_id,datagouv_resource_id,datagouv_organization_or_owner,created_at,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified
0,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,+33 (1) 49939011,Asia Automotive,ATHTBE1012061,ATHTBE1012061,Asia Automotive,Voirie,"3 Rue Laurent Lavoisier, 80330 Longueau",NaN,"[2.36761800,49.87723700]",3,ATHTBE1012061,ATHTBE1012061,22.0,false,true,false,false,false,false,true,true,NaN,NaN,Accès réservé,false,24/7,Accessibilité inconnue,inconnu,false,NaN,NaN,2022-03-22,EF connector is available at the site separately,2026-08-31,false,2026-08-31T19:00:31.772000+00:00,64060c2ac773dcf3fabbe5d2,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,2.367618,49.877237,NaN,NaN,False,False,False
1,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,+33 (1) 49939011,Asia Automotive,ATHTBE1012062,ATHTBE1012062,Asia Automotive,Voirie,"3 Rue Laurent Lavoisier, 80330 Longueau",NaN,"[2.36761800,49.87723700]",3,ATHTBE1012062,ATHTBE1012062,22.0,false,true,false,false,false,false,true,true,NaN,NaN,Accès réservé,false,24/7,Accessibilité inconnue,inconnu,false,NaN,NaN,2022-03-22,EF connector is available at the site separately,2026-08-31,false,2026-08-31T19:00:31.772000+00:00,64060c2ac773dcf3fabbe5d2,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,2.367618,49.877237,NaN,NaN,False,False,False
2,ASIA AUTOMOTIVE,892613597.0,laurent@lanie.fr,RAIDEN,contact@chargeguru.com,188246000,ASIA AUTOMOTIVE,Non concerné,NaN,ASIA AUTOMOTIVE,Parking public,3 Rue Laurent Lavoisier 80330 Longueau,80489,"[2.3673462, 49.8762485]",2,ATHTBE1012062,NaN,44.0,TRUE,TRUE,FALSE,FALSE,FALSE,NaN,FALSE,NaN,NaN,NaN,Accès libre,FALSE,24/7,Accessibilité inconnue,Inconnue,FALSE,NaN,NaN,2021-10-13,NaN,2021-10-13,NaN,2026-04-20T12:11:36.189000+00:00,63f38ce81be1f6867a668b51,5be52fb0-72ca-43c6-b88d-0d56fb3efff6,raiden-sas,2025-10-07T09:11:00.465000+00:00,2.367346,49.876249,80330.0,Longueau,True,True,False



=== TYPES ET COMPLÉTUDE ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226963 entries, 0 to 226962
Data columns (total 52 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   nom_amenageur                        225550 non-null  object 
 1   siren_amenageur                      152254 non-null  float64
 2   contact_amenageur                    162437 non-null  object 
 3   nom_operateur                        226489 non-null  object 
 4   contact_operateur                    226963 non-null  object 
 5   telephone_operateur                  170889 non-null  object 
 6   nom_enseigne                         226963 non-null  object 
 7   id_station_itinerance                226963 non-null  object 
 8   id_station_local                     167986 non-null  object 
 9   nom_station                          226963 non-null  object 
 10  implantation_station                 226963 non-nul

,siren_amenageur,nbre_pdc,puissance_nominale,consolidated_longitude,consolidated_latitude,consolidated_code_postal
count,1.522540e+05,226963.000000,226963.000000,226963.000000,226963.000000,138452.000000
mean,6.827217e+08,14.322405,101.360157,2.718321,46.743656,52746.875040
std,2.636241e+08,44.753228,579.957086,4.545096,4.037386,26729.613408
min,0.000000e+00,1.000000,0.000000,-149.905377,-44.996198,1000.000000
25%,4.508892e+08,2.000000,22.000000,0.993870,44.880730,31210.000000
50%,8.427185e+08,4.000000,22.000000,2.449010,47.355382,57245.000000
75%,8.951636e+08,9.000000,100.000000,4.851826,48.858112,76600.000000
max,9.921637e+08,505.000000,160000.000000,166.462000,61.520355,97418.000000



=== SYNTHÈSE DES VALEURS MANQUANTES ===


,nuls,pourcentage
observations,177193,78.07
tarification,171112,75.39
cable_t2_attache,111855,49.28
num_pdl,103699,45.69
date_mise_en_service,93111,41.02
consolidated_code_postal,88511,39.00
id_pdc_local,82033,36.14
siren_amenageur,74709,32.92
consolidated_commune,70808,31.20
raccordement,70034,30.86



=== DIAGNOSTIC DES DOUBLONS ===
1. Doublons stricts (lignes entièrement identiques) : 0
2. Doublons sur l'identifiant PDC (id_pdc_itinerance) : 60105
3. Nombre de stations uniques (id_station_itinerance) : 57,240


In [13]:
# Fonction de sélection des colonnes et dédoublonnage

def select_and_deduplicate(df: pd.DataFrame) -> pd.DataFrame:
    """Filtre les colonnes d'intérêt, supprime les doublons stricts

    et ne conserve que la dernière mise à jour par Point de Charge (PDC).
    """
    cols_to_keep = [
        "id_station_itinerance",
        "id_pdc_itinerance",
        "nom_station",
        "nom_operateur",
        "nom_amenageur",
        "adresse_station",
        "code_insee_commune",
        "consolidated_code_postal",
        "consolidated_commune",
        "consolidated_latitude",
        "consolidated_longitude",
        "puissance_nominale",
        "nbre_pdc",
        "implantation_station",
        "condition_acces",
        "prise_type_2",
        "prise_type_combo_ccs",
        "last_modified",  # Conservé temporairement pour le tri du dédoublonnage
    ]

    available_cols = [c for c in cols_to_keep if c in df.columns]
    df_filtered = df[available_cols].copy()

    # Suppression des doublons stricts (lignes 100 % identiques)
    df_filtered = df_filtered.drop_duplicates()

    # Dédoublonnage sur id_pdc_itinerance en gardant la version la plus récente
    if (
        "id_pdc_itinerance" in df_filtered.columns
        and "last_modified" in df_filtered.columns
    ):
        df_filtered["last_modified"] = pd.to_datetime(
            df_filtered["last_modified"], errors="coerce"
        )
        df_filtered = df_filtered.sort_values(
            by="last_modified", ascending=True
        )
        df_filtered = df_filtered.drop_duplicates(
            subset=["id_pdc_itinerance"], keep="last"
        )

    # Suppression de la colonne temporaire de date
    if "last_modified" in df_filtered.columns:
        df_filtered = df_filtered.drop(columns=["last_modified"])

    return df_filtered


# Exécution de la première étape
df_step1 = select_and_deduplicate(df_raw)

print(f"Dimensions après sélection et dédoublonnage : {df_step1.shape}")
df_step1.to_csv("../data/processed/dataset_irve_step1.csv", index=False)

Dimensions après sélection et dédoublonnage : (166858, 17)


## 1. Sélection des variables et dédoublonnage (`dataset_step1`)

Cette première phase de préparation permet d'extraire la structure utile du jeu de données brut et de garantir l'unicité des points de charge, tout en évitant le chevauchement de données (datalakage) pour les futures étapes de Machine Learning.

### Actions réalisées :
* **Sélection stratégique (17 colonnes)** : Conservation uniquement des identifiants métiers (`id_station_itinerance`, `id_pdc_itinerance`), des localisations (`latitude`, `longitude`, `code_insee_commune`, `consolidated_code_postal`, `consolidated_commune`), des caractéristiques techniques (`puissance_nominale`, `nbre_pdc`, types de prises) et des acteurs (`nom_station`, `nom_operateur`, `nom_amenageur`).
* **Suppression des doublons stricts** : Élimination des lignes 100 % identiques issues de la centralisation des fichiers sources.
* **Dédoublonnage métier par Point de Charge (PDC)** : Conversion du champ `last_modified` en horodatage, tri chronologique, puis conservation exclusive de la mise à jour la plus récente pour chaque `id_pdc_itinerance`.

### Résultat :
* **Volumétrie** : Passage de 226 963 lignes brutes à **166 858 points de charge uniques**.
* **Export intermédiaire** : `data/processed/dataset_irve_step1.csv`

In [4]:
# ÉTAPES 1 & 2 : NETTOYAGE GÉOSPATIAL, HARMONISATION PUISSANCE ET FEATURE ENGINEERING

df_clean = df_step1.copy()

# ==============================================================================
# ÉTAPE 1 : NETTOYAGE GÉOSPATIAL ET RÈGLES DE SUPPRESSION
# ==============================================================================

# Remplacement des placeholders par NaN
placeholders = [
    "non concerné",
    "non concerne",
    "n/a",
    "nr",
    "null",
    "inconnu",
    "nc",
    "0",
    "00000",
    "-",
    "none",
    "nan",
    "",
]
text_cols = df_clean.select_dtypes(include="object").columns

for col in text_cols:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    mask_ph = df_clean[col].str.lower().isin(placeholders)
    df_clean.loc[mask_ph, col] = np.nan

# Invalidation des coordonnées GPS hors bornes (France Hexagonale + DROM-COM)
mask_lat_invalide = (df_clean["consolidated_latitude"] < -22.5) | (
    df_clean["consolidated_latitude"] > 51.5
)
df_clean.loc[
    mask_lat_invalide, ["consolidated_latitude", "consolidated_longitude"]
] = np.nan

# Règle de suppression : Ni GPS Ni Adresse
mask_sans_gps = (
    df_clean["consolidated_latitude"].isna()
    | df_clean["consolidated_longitude"].isna()
)
mask_sans_adresse = df_clean["adresse_station"].isna()

mask_inexploitable = mask_sans_gps & mask_sans_adresse
nb_suppr = mask_inexploitable.sum()

df_clean = df_clean[~mask_inexploitable].copy()
print(f"Lignes inexploitables supprimées (Ni GPS, Ni adresse) : {nb_suppr}")

# ==============================================================================
# ÉTAPE 2 : HARMONISATION DE LA PUISSANCE ET FEATURE ENGINEERING
# ==============================================================================

# Correction d'échelle (Watts -> kW si > 500 kW)
mask_watts = df_clean["puissance_nominale"] > 500
df_clean.loc[mask_watts, "puissance_nominale"] = (
    df_clean.loc[mask_watts, "puissance_nominale"] / 1000
)

# Invalidation des puissances irréalistes (< 1 kW) et imputation
df_clean.loc[df_clean["puissance_nominale"] < 1.0, "puissance_nominale"] = (
    np.nan
)

median_puissance_valide = df_clean.loc[
    df_clean["puissance_nominale"] >= 1.0, "puissance_nominale"
].median()
df_clean["puissance_nominale"] = df_clean["puissance_nominale"].fillna(
    median_puissance_valide
)

# Correction de nbre_pdc (calcul uniquement sur les identifiants valides)
mask_id_valide = df_clean["id_station_itinerance"].notna()
pdc_real_counts = df_clean[mask_id_valide].groupby("id_station_itinerance")[
    "id_pdc_itinerance"
].transform("count")

df_clean.loc[mask_id_valide, "nbre_pdc"] = pdc_real_counts
df_clean["nbre_pdc"] = df_clean["nbre_pdc"].fillna(1).clip(upper=30)

# Feature Engineering : Tranches de puissance (Target ML)
bins = [-np.inf, 7, 22, 150, np.inf]
labels = [
    "Lente (<7kW)",
    "Accélérée (7-22kW)",
    "Rapide (22-150kW)",
    "Ultra-rapide (>150kW)",
]
df_clean["tranche_puissance"] = pd.cut(
    df_clean["puissance_nominale"], bins=bins, labels=labels
)

# Conversion des types de prises en booléens
cols_prises = ["prise_type_2", "prise_type_combo_ccs"]
for col in cols_prises:
    if col in df_clean.columns:
        df_clean[col] = (
            df_clean[col]
            .astype(str)
            .str.lower()
            .isin(["true", "1", "1.0", "oui"])
        )

print("✅ Étapes 1 et 2 combinées, validées et appliquées.")

Lignes inexploitables supprimées (Ni GPS, Ni adresse) : 0
✅ Étapes 1 et 2 combinées, validées et appliquées.


In [5]:
# Géocodage, imputation spatiale et enrichissement géographique

print("=================================================================")
print("1. TÉLÉCHARGEMENT AUTOMATIQUE DU RÉFÉRENTIEL GÉOGRAPHIQUE")
print("=================================================================")

FICHIER_LOCAL = "referentiel_communes.csv"
URL_OPENDATASOFT = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/georef-france-commune/exports/csv?lang=fr&timezone=Europe%2FBerlin&delimiter=%3B"

if not os.path.exists(FICHIER_LOCAL):
    print(" Téléchargement du référentiel (communes, codes postaux, GPS)...")
    res = requests.get(
        URL_OPENDATASOFT, headers={"User-Agent": "Mozilla/5.0"}, timeout=60
    )
    res.raise_for_status()

    with open(FICHIER_LOCAL, "wb") as f:
        f.write(res.content)
    print("✅ Référentiel téléchargé et sauvegardé sous 'referentiel_communes.csv'.")
else:
    print("✅ Référentiel local trouvé.")

# Chargement du fichier
ref_geo = pd.read_csv(FICHIER_LOCAL, sep=";", dtype=str)

# Détection et renommage automatique des colonnes clés
col_insee = next(c for c in ref_geo.columns if "com_code" in c or "insee" in c.lower())
col_nom = next(c for c in ref_geo.columns if "com_name" in c or "nom" in c.lower())
col_cp = next(c for c in ref_geo.columns if "postal" in c.lower() or "cp" in c.lower() or "com_current_code" in c)
col_geo = next(c for c in ref_geo.columns if "geo_point" in c.lower() or "coord" in c.lower())

ref_geo = ref_geo.rename(columns={
    col_insee: "code_insee_ref",
    col_nom: "commune_ref",
    col_cp: "cp_ref",
    col_geo: "geo_point"
})

# Extraction des coordonnées lat/lon
ref_geo[["lat_ref", "lon_ref"]] = ref_geo["geo_point"].str.split(",", expand=True).astype(float)

# Tables optimisées pour les jointures
ref_geo_unique = ref_geo.drop_duplicates(subset=["code_insee_ref"]).copy()
ref_knn = ref_geo.dropna(subset=["lat_ref", "lon_ref"]).reset_index(drop=True)

print(f"Communes chargées : {len(ref_geo_unique)}")
print(f"Points GPS valides : {len(ref_knn)}")

print("\n=================================================================")
print("2. GÉOCODAGE & IMPUTATION (CODES POSTAUX ET COMMUNES)")
print("=================================================================")

# Passe 1 : jointure directe par Code INSEE
df_clean = df_clean.merge(
    ref_geo_unique[["code_insee_ref", "cp_ref", "commune_ref"]],
    left_on="code_insee_commune",
    right_on="code_insee_ref",
    how="left"
)

df_clean["consolidated_code_postal"] = df_clean["consolidated_code_postal"].fillna(df_clean["cp_ref"])
df_clean["consolidated_commune"] = df_clean["consolidated_commune"].fillna(df_clean["commune_ref"])
df_clean.drop(columns=["code_insee_ref", "cp_ref", "commune_ref"], inplace=True, errors="ignore")

# Passe 2 : proximité géographique BallTree (points orphelins)
mask_orphelins = (
    df_clean["consolidated_code_postal"].isna() &
    df_clean["consolidated_latitude"].notna() &
    df_clean["consolidated_longitude"].notna()
)

nb_orphelins = mask_orphelins.sum()
if nb_orphelins > 0 and len(ref_knn) > 0:
    ref_coords_rad = np.radians(ref_knn[["lat_ref", "lon_ref"]].values)
    points_orphelins_rad = np.radians(
        df_clean.loc[mask_orphelins, ["consolidated_latitude", "consolidated_longitude"]].values
    )

    tree = BallTree(ref_coords_rad, metric="haversine")
    _, indices = tree.query(points_orphelins_rad, k=1)
    indices_flat = indices.flatten()

    df_clean.loc[mask_orphelins, "consolidated_code_postal"] = ref_knn.loc[indices_flat, "cp_ref"].values
    df_clean.loc[mask_orphelins, "consolidated_commune"] = ref_knn.loc[indices_flat, "commune_ref"].values
    df_clean.loc[mask_orphelins, "code_insee_commune"] = ref_knn.loc[indices_flat, "code_insee_ref"].values

print(f"• Points orphelins géocodés par proximité GPS : {nb_orphelins}")

print("\n=================================================================")
print("3. EXTRACTION STRUCTURÉE DU CODE DÉPARTEMENTAL")
print("=================================================================")

def extraire_departement(row):
    insee = str(row["code_insee_commune"]).strip().zfill(5) if pd.notna(row["code_insee_commune"]) else ""
    cp = str(row["consolidated_code_postal"]).strip().zfill(5) if pd.notna(row["consolidated_code_postal"]) else ""

    if len(insee) == 5:
        if insee.startswith(("2A", "2B")):
            return insee[:2]
        if insee.startswith(("97", "98")):
            return insee[:3]
        return insee[:2]

    if len(cp) == 5:
        if cp.startswith("202"):
            return "2B"
        if cp.startswith(("200", "201", "20")):
            return "2A"
        if cp.startswith(("97", "98")):
            return cp[:3]
        return cp[:2]

    return None

df_clean["code_departement"] = df_clean.apply(extraire_departement, axis=1)

print("\n=================================================================")
print("4. BILAN DU NETTOYAGE GÉOGRAPHIQUE")
print("=================================================================")
print(f"• Total lignes           : {len(df_clean)}")
print(f"• Codes postaux manquants : {df_clean['consolidated_code_postal'].isna().sum()}")
print(f"• Communes manquantes     : {df_clean['consolidated_commune'].isna().sum()}")
print(f"• Départements identifiés : {df_clean['code_departement'].nunique()}")

print("\n--- Échantillon de résultat ---")
cols_preview = ["nom_station", "consolidated_latitude", "consolidated_longitude", "code_insee_commune", "consolidated_code_postal", "consolidated_commune", "code_departement"]
print(df_clean[cols_preview].head(5).to_string(index=False))

1. TÉLÉCHARGEMENT AUTOMATIQUE DU RÉFÉRENTIEL GÉOGRAPHIQUE
✅ Référentiel local trouvé.
Communes chargées : 34888
Points GPS valides : 34888

2. GÉOCODAGE & IMPUTATION (CODES POSTAUX ET COMMUNES)
• Points orphelins géocodés par proximité GPS : 58540

3. EXTRACTION STRUCTURÉE DU CODE DÉPARTEMENTAL

4. BILAN DU NETTOYAGE GÉOGRAPHIQUE
• Total lignes           : 166858
• Codes postaux manquants : 0
• Communes manquantes     : 0
• Départements identifiés : 102

--- Échantillon de résultat ---
                                                nom_station  consolidated_latitude  consolidated_longitude code_insee_commune consolidated_code_postal consolidated_commune code_departement
    Metropolis - ePremium - Neuilly-sur-Seine - Bretteville              48.875794                2.255578              92051                  92200.0    Neuilly-sur-Seine               92
Metropolis - ePremium - Neuilly-sur-Seine - Achille Peretti              48.885127                2.265676              92051      

In [6]:
# Fonction de nettoyage strict des codes postaux
def nettoyer_cp(val):
    if pd.isna(val) or val == "":
        return None
    s = str(val).split(".")[0].strip()
    return s.zfill(5) if len(s) <= 5 else s


df_clean["consolidated_code_postal"] = df_clean[
    "consolidated_code_postal"
].apply(nettoyer_cp)


# Recalcul des départements sur la base des CP propres
def extraire_dep_propre(row):
    cp = (
        str(row["consolidated_code_postal"])
        if pd.notna(row["consolidated_code_postal"])
        else ""
    )
    insee = (
        str(row["code_insee_commune"]).split(".")[0].zfill(5)
        if pd.notna(row["code_insee_commune"])
        else ""
    )

    if len(cp) == 5:
        if cp.startswith("202"):
            return "2B"
        if cp.startswith(("200", "201", "20")):
            return "2A"
        if cp.startswith(("97", "98")):
            return cp[:3]
        return cp[:2]

    if len(insee) == 5:
        if insee.startswith(("2A", "2B")):
            return insee[:2]
        if insee.startswith(("97", "98")):
            return insee[:3]
        return insee[:2]

    return None


df_clean["code_departement"] = df_clean.apply(extraire_dep_propre, axis=1)

# Vérification des manquants
print("--- Bilan après correction ---")
print(
    f"• Codes postaux manquants : {df_clean['consolidated_code_postal'].isna().sum()}"
)
print(
    f"• Départements manquants : {df_clean['code_departement'].isna().sum()}"
)

# Aperçu
df_clean[
    [
        "nom_station",
        "consolidated_commune",
        "consolidated_code_postal",
        "code_departement",
    ]
].head(5)

# Téléchargement du dataframe

os.makedirs("../data/processed", exist_ok=True)
df_clean.to_csv("../data/processed/dataset_irve_clean.csv", index=False)
print("✅ Fichier final sauvegardé dans data/processed/dataset_irve_clean.csv")

--- Bilan après correction ---
• Codes postaux manquants : 0
• Départements manquants : 0
✅ Fichier final sauvegardé dans data/processed/dataset_irve_clean.csv


In [7]:
# Liste des colonnes de type 'object'
cols_object = df_clean.select_dtypes(include=["object"]).columns.tolist()
print("Colonnes au format texte :", cols_object)

# Aperçu des premières valeurs de ces colonnes
df_clean[cols_object].head(3)

Colonnes au format texte : ['id_station_itinerance', 'id_pdc_itinerance', 'nom_station', 'nom_operateur', 'nom_amenageur', 'adresse_station', 'code_insee_commune', 'consolidated_code_postal', 'consolidated_commune', 'implantation_station', 'condition_acces', 'code_departement']


,id_station_itinerance,id_pdc_itinerance,nom_station,nom_operateur,nom_amenageur,adresse_station,code_insee_commune,consolidated_code_postal,consolidated_commune,implantation_station,condition_acces,code_departement
0,FRMGPP92051A,FRMGPE92051AB1P2,Metropolis - ePremium - Neuilly-sur-Seine - Br...,SPIE CITYNETWORKS,METROPOLIS,2 Avenue de Bretteville 92200 NEUILLY-SUR-SEIN...,92051,92200,Neuilly-sur-Seine,Voirie,Accès libre,92
1,FRMGPP92051E,FRMGPE92051EB1P2,Metropolis - ePremium - Neuilly-sur-Seine - Ac...,SPIE CITYNETWORKS,METROPOLIS,163 Avenue Achille Peretti 92200 NEUILLY-SUR-S...,92051,92200,Neuilly-sur-Seine,Voirie,Accès libre,92
2,FRMGPP92051F,FRMGPE92051FB2P2,Metropolis - ePremium - Neuilly-sur-Seine - Mo...,SPIE CITYNETWORKS,METROPOLIS,19 Rue Montrosier 92200 NEUILLY-SUR-SEINE (92),92051,92200,Neuilly-sur-Seine,Voirie,Accès libre,92


# Documentation du pipeline : nettoyage, imputation spatiale et feature engineering

## 1. Nettoyage géospatial, harmonisation des données et feature engineering

Cette phase valide la cohérence des coordonnées géographiques, harmonise les métriques de puissance et prépare les variables cibles pour les futurs modèles de Machine Learning.

### Actions réalisées
* **Standardisation des placeholders** : identification des chaînes de caractères parasites (`"inconnu"`, `"n/a"`, `"00000"`, `"-"`, etc.) et conversion explicite en `NaN` sur toutes les colonnes textuelles.
* **Filtrage des coordonnées GPS aberrantes** : invalidation des latitudes/longitudes situées en dehors des limites géographiques de la France métropolitaine et des DROM-COM (hors de l'intervalle `[-22,5 ; 51,5]`).
* **Règle d'exclusion stricte** : suppression des données inexploitables, définies par l'absence simultanée de coordonnées GPS valides et d'adresse physique.
* **Harmonisation de la puissance nominale (`puissance_nominale`)** :
  * Correction d'échelle : conversion automatique des valeurs exprimées en Watts (> 500 kW) vers des kilowatts (kW).
  * Invalidation des puissances irréalistes (< 1 kW) et imputation par la médiane des valeurs valides.
* **Recalcul du nombre de points de charge (`nbre_pdc`)** : reconstitution dynamique du nombre réel de bornes par station (`id_station_itinerance`) au lieu de se fier aux valeurs déclaratives, avec un plafonnement technique à 30 bornes.
* **Feature Engineering** :
  * Création de la variable catégorielle **`tranche_puissance`** répartie en 4 classes métiers : *Lente (<7 kW)*, *Accélérée (7-22 kW)*, *Rapide (22-150 kW)* et *Ultra-rapide (>150 kW)*.
  * Typage booléen strict des variables de prises (`prise_type_2`, `prise_type_combo_ccs`).

---

## 2. Géocodage, imputation spatiale par BallTree et enrichissement géographique

L'objectif de cette étape est de garantir la complétude géodésique du dataset en réparant les codes postaux, les communes et les départements manquants.

### Méthodologie
1. **Intégration d'un référentiel géographique officiel** : téléchargement et structuration de la base OpenDataSoft comprenant **34 888 communes françaises** enrichies de leurs codes INSEE, codes postaux et centroïdes GPS.
2. **Passe 1 — Jointure déterministe** : imputation des codes postaux et noms de communes par correspondance exacte du code INSEE (`code_insee_commune`).
3. **Passe 2 — Imputation géospatiale KNN (BallTree)** : pour les bornes orphelines disposant de coordonnées GPS mais sans code postal, exécution d'un algorithme de plus proches voisins (*K-Nearest Neighbors*) basé sur la métrique *Haversine* appliquée aux coordonnées converties en radians.
4. **Passe 3 — Extraction du code départemental (`code_departement`)** : traitement algorithmique dédié gérant les spécificités administratives :
   * Corse (2A / 2B d'après INSEE ou codes postaux 200xx-202xx).
   * DROM-COM (codes à 3 chiffres de type 97x / 98x).
   * Métropole (codes à 2 chiffres).

---

## 3. Bilan synthétique de la qualité des données

Le tableau ci-dessous résume l'impact des différentes phases du pipeline sur la qualité et la volumétrie du jeu de données :

| Métrique de contrôle | État initial (Brut) | Après dédoublonnage | Bilan final (Après géocodage) |
| :--- | :---: | :---: | :---: |
| **Volumétrie (Lignes / PDC)** | 226 963 | 166 858 | **166 858** |
| **Points de charge uniques** | Non garantis | 100 % uniques | **100 % uniques** |
| **Codes postaux manquants** | 88 511 (39,0 %) | — | **0 (0,0 %)** |
| **Communes manquantes** | 70 808 (31,2 %) | — | **0 (0,0 %)** |
| **Points orphelins sauvés (KNN)** | — | — | **58 540** |
| **Départements identifiés** | Incomplets | — | **102 (100 % du territoire)** |

**Conclusion de l'étape** : Le jeu de données est désormais fully-cleaned, géodésiquement complet, sans aucune donnée manquante sur les variables géographiques clés, et prêt pour l'ingestion dans la base SQLite `irve_database.db`.